In [36]:
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [37]:
PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "models").exists():
    PROJECT_ROOT = Path.cwd()

MODEL_DIR = PROJECT_ROOT / "models"

CWRU_PROCESSED = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "CWRU"
)

WINDOW_DIR = (
    CWRU_PROCESSED
    / "windows"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
)

print("Project root:", PROJECT_ROOT)
print("Model directory:", MODEL_DIR)
print("CWRU directory:", CWRU_PROCESSED)

Project root: c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection
Model directory: c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\models
CWRU directory: c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU


In [38]:
FP32_MODEL_PATH = (
    MODEL_DIR / "vaac_tiny_best.keras"
)

print("Model exists:",
      FP32_MODEL_PATH.exists())

print("Model path:",
      FP32_MODEL_PATH)

Model exists: True
Model path: c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\models\vaac_tiny_best.keras


In [39]:
fp32_model = tf.keras.models.load_model(
    FP32_MODEL_PATH
)

print("Model loaded successfully.")

print("Input shape:",
      fp32_model.input_shape)

print("Output shape:",
      fp32_model.output_shape)

print("Parameters:",
      fp32_model.count_params())

Model loaded successfully.
Input shape: (None, 12000, 1)
Output shape: (None, 4)
Parameters: 4244


In [40]:
print("Last layer:")
print(fp32_model.layers[-1])

Last layer:
<Dense name=classification_output, built=True>


In [41]:
print("Last layer name:",
      fp32_model.layers[-1].name)

print("Last layer activation:",
      fp32_model.layers[-1].activation)

Last layer name: classification_output
Last layer activation: <function softmax at 0x000001B15D48C310>


In [42]:
TEST_METADATA = (
    WINDOW_DIR /
    "test_metadata.csv"
)

test_df = pd.read_csv(
    TEST_METADATA
)

print("Test shape:",
      test_df.shape)

print("\nClass distribution:")
print(test_df["class"].value_counts())

Test shape: (136, 8)

Class distribution:
class
Healthy       79
Ball          19
Inner Race    19
Outer Race    19
Name: count, dtype: int64


In [43]:
possible_files = [
    *MODEL_DIR.rglob("*"),
    *RESULTS_DIR.rglob("*"),
    *CWRU_PROCESSED.rglob("*")
]

mapping_candidates = []

for path in possible_files:
    if path.is_file():
        name = path.name.lower()
        
        if any(
            term in name
            for term in [
                "class",
                "label",
                "mapping",
                "history",
                "training",
                "prediction"
            ]
        ):
            mapping_candidates.append(path)

print("Possible mapping/training files:")

for path in mapping_candidates:
    print(path)

Possible mapping/training files:
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results\quantized\fp32_vs_int8_test_predictions.csv
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results\vaac_tiny\temporal_consensus_predictions.csv
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results\vaac_tiny\test_predictions.csv
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results\vaac_tiny\training_history.csv
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\class_feature_means.csv


In [44]:
print(test_df.iloc[0])

recording_id    B007_3_X121
source_file      B007_3.mat
signal_id              X121
class                  Ball
window_id                 0
start_sample              0
end_sample            12000
label                     1
Name: 0, dtype: object


In [45]:
all_npy_files = list(
    CWRU_PROCESSED.rglob("*.npy")
)

window_file_index = {
    path.name: path
    for path in all_npy_files
}

print("Total indexed .npy files:",
      len(window_file_index))

Total indexed .npy files: 583


In [46]:
def get_window_path(row):
    
    filename = (
        f"{row['recording_id']}"
        f"_window_{int(row['window_id']):04d}.npy"
    )
    
    if filename not in window_file_index:
        raise FileNotFoundError(
            filename
        )
    
    return window_file_index[filename]

In [47]:
row = test_df.iloc[0]

path = get_window_path(row)

signal = np.load(path).astype(
    np.float32
)

signal = signal.reshape(
    1, 12000, 1
)

raw_output = fp32_model.predict(
    signal,
    verbose=0
)[0]

print("True class:")
print(row["class"])

print("\nRaw model output:")
print(raw_output)

print("\nArgmax:")
print(np.argmax(raw_output))

True class:
Ball

Raw model output:
[1.2437244e-04 8.3114700e-03 3.2581473e-05 9.9153155e-01]

Argmax:
3


In [48]:
print(
    "Output sum:",
    np.sum(raw_output)
)

Output sum: 1.0


In [49]:
print(
    "Minimum output:",
    np.min(raw_output)
)

print(
    "Maximum output:",
    np.max(raw_output)
)

Minimum output: 3.2581473e-05
Maximum output: 0.99153155


In [50]:
for i, value in enumerate(raw_output):
    print(
        f"Output index {i}: "
        f"{value:.8f}"
    )

Output index 0: 0.00012437
Output index 1: 0.00831147
Output index 2: 0.00003258
Output index 3: 0.99153155


In [51]:
model_index = int(
    np.argmax(raw_output)
)

true_class = row["class"]

print("True class:", true_class)
print("Model output index:", model_index)

True class: Ball
Model output index: 3


In [52]:
print(
    test_df.groupby("class").size()
)

class
Ball          19
Healthy       79
Inner Race    19
Outer Race    19
dtype: int64


In [53]:
class_examples = (
    test_df
    .groupby("class", group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

display(
    class_examples[
        [
            "recording_id",
            "source_file",
            "signal_id",
            "class",
            "window_id"
        ]
    ]
)

,recording_id,source_file,signal_id,class,window_id
0,B007_3_X121,B007_3.mat,X121,Ball,0
1,IR007_3_X108,IR007_3.mat,X108,Inner Race,0
2,OR007@6_3_X133,OR007@6_3.mat,X133,Outer Race,0
3,NORMAL_3_X100,NORMAL_3.mat,X100,Healthy,0


In [54]:
mapping_observations = []

for _, row in class_examples.iterrows():
    
    path = get_window_path(row)
    
    signal = np.load(path).astype(
        np.float32
    )
    
    signal = signal.reshape(
        1, 12000, 1
    )
    
    output = fp32_model.predict(
        signal,
        verbose=0
    )[0]
    
    predicted_index = int(
        np.argmax(output)
    )
    
    mapping_observations.append({
        "true_class": row["class"],
        "model_output_index": predicted_index,
        "confidence": float(
            np.max(output)
        )
    })

mapping_df = pd.DataFrame(
    mapping_observations
)

display(mapping_df)

,true_class,model_output_index,confidence
0,Ball,3,0.991532
1,Inner Race,1,0.831189
2,Outer Race,3,0.997230
3,Healthy,3,0.978726


In [55]:
class_mapping_test = []

for class_name in sorted(
    test_df["class"].unique()
):
    
    subset = test_df[
        test_df["class"] == class_name
    ].head(10)
    
    for _, row in subset.iterrows():
        
        path = get_window_path(row)
        
        signal = np.load(
            path
        ).astype(np.float32)
        
        signal = signal.reshape(
            1, 12000, 1
        )
        
        output = fp32_model.predict(
            signal,
            verbose=0
        )[0]
        
        predicted_index = int(
            np.argmax(output)
        )
        
        class_mapping_test.append({
            "true_class": class_name,
            "model_output_index":
                predicted_index,
            "confidence":
                float(np.max(output))
        })

mapping_test_df = pd.DataFrame(
    class_mapping_test
)

display(mapping_test_df)

,true_class,model_output_index,confidence
0,Ball,3,0.991532
1,Ball,3,0.991519
2,Ball,3,0.991489
3,Ball,3,0.991643
4,Ball,3,0.991601
5,Ball,3,0.991387
6,Ball,3,0.991524
7,Ball,3,0.991909
8,Ball,3,0.992053
9,Ball,3,0.991958


In [56]:
dominant_mapping = (
    mapping_test_df
    .groupby("true_class")[
        "model_output_index"
    ]
    .agg(
        lambda x: x.value_counts().idxmax()
    )
)

print(
    "Observed model output mapping:"
)

print(dominant_mapping)

Observed model output mapping:
true_class
Ball          3
Healthy       3
Inner Race    1
Outer Race    3
Name: model_output_index, dtype: int64


In [57]:
mapping_consistency = (
    mapping_test_df
    .groupby("true_class")[
        "model_output_index"
    ]
    .value_counts()
)

print(
    mapping_consistency
)

true_class  model_output_index
Ball        3                     10
Healthy     3                     10
Inner Race  1                     10
Outer Race  3                     10
Name: count, dtype: int64


In [58]:
observed_class_to_index = {}

for class_name in mapping_test_df[
    "true_class"
].unique():
    
    subset = mapping_test_df[
        mapping_test_df["true_class"]
        == class_name
    ]
    
    dominant_index = (
        subset["model_output_index"]
        .value_counts()
        .idxmax()
    )
    
    observed_class_to_index[
        class_name
    ] = int(dominant_index)

print(
    "Observed class → model index mapping:"
)

for class_name, index in (
    observed_class_to_index.items()
):
    print(
        f"{class_name} -> {index}"
    )

Observed class → model index mapping:
Ball -> 3
Healthy -> 3
Inner Race -> 1
Outer Race -> 3


In [59]:
indices = list(
    observed_class_to_index.values()
)

print("Observed indices:", indices)

print(
    "Unique indices:",
    set(indices)
)

print(
    "One-to-one mapping:",
    len(set(indices)) == 4
)

Observed indices: [3, 3, 1, 3]
Unique indices: {1, 3}
One-to-one mapping: False


In [60]:
y_true_observed = []
y_pred_observed = []

for _, row in test_df.iterrows():
    
    path = get_window_path(row)
    
    signal = np.load(
        path
    ).astype(np.float32)
    
    signal = signal.reshape(
        1, 12000, 1
    )
    
    output = fp32_model.predict(
        signal,
        verbose=0
    )[0]
    
    predicted_index = int(
        np.argmax(output)
    )
    
    y_true_observed.append(
        observed_class_to_index[
            row["class"]
        ]
    )
    
    y_pred_observed.append(
        predicted_index
    )

In [61]:
observed_accuracy = accuracy_score(
    y_true_observed,
    y_pred_observed
)

print(
    "FP32 accuracy using observed mapping:",
    f"{observed_accuracy:.4f}"
)

FP32 accuracy using observed mapping: 1.0000


In [62]:
cm_corrected = confusion_matrix(
    y_true_observed,
    y_pred_observed,
    labels=[0, 1, 2, 3]
)

print(
    "Corrected FP32 confusion matrix:"
)

print(cm_corrected)

Corrected FP32 confusion matrix:
[[  0   0   0   0]
 [  0  19   0   0]
 [  0   0   0   0]
 [  0   0   0 117]]


In [63]:
print(
    classification_report(
        y_true_observed,
        y_pred_observed,
        labels=[0, 1, 2, 3],
        zero_division=0
    )
)

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      1.00      1.00        19
           2       0.00      0.00      0.00         0
           3       1.00      1.00      1.00       117

    accuracy                           1.00       136
   macro avg       0.50      0.50      0.50       136
weighted avg       1.00      1.00      1.00       136



In [64]:
print("=" * 65)
print("STEP 226 — OUTPUT MAPPING DIAGNOSIS")
print("=" * 65)

print("\nModel:")
print("VAAC-Tiny")

print("\nParameters:")
print(fp32_model.count_params())

print("\nTest windows:")
print(len(test_df))

print("\nObserved class-to-output mapping:")

for class_name, index in (
    observed_class_to_index.items()
):
    print(
        f"  {class_name:<15} -> "
        f"Output {index}"
    )

print("\nOne-to-one mapping:",
      len(set(indices)) == 4)

print(
    "\nFP32 accuracy using observed mapping:",
    f"{observed_accuracy:.4f}"
)

print("\nStep 226 diagnosis completed.")

STEP 226 — OUTPUT MAPPING DIAGNOSIS

Model:
VAAC-Tiny

Parameters:
4244

Test windows:
136

Observed class-to-output mapping:
  Ball            -> Output 3
  Healthy         -> Output 3
  Inner Race      -> Output 1
  Outer Race      -> Output 3

One-to-one mapping: False

FP32 accuracy using observed mapping: 1.0000

Step 226 diagnosis completed.


In [65]:
# Step 226.23 — Search project files for label mapping information

search_files = []

for root in [
    PROJECT_ROOT / "notebooks",
    PROJECT_ROOT / "models",
    PROJECT_ROOT / "results",
    PROJECT_ROOT / "data" / "processed" / "CWRU"
]:
    
    if root.exists():
        for path in root.rglob("*"):
            if path.is_file():
                search_files.append(path)

print("Files available for inspection:", len(search_files))

for path in search_files:
    if path.suffix.lower() in [".csv", ".json", ".txt", ".py", ".ipynb"]:
        print(path)

Files available for inspection: 1244
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\notebooks\01_cwru_exploration.ipynb
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\notebooks\02_cwru_metadata.ipynb
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\notebooks\03_cwru_signal_extraction.ipynb
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\notebooks\04_cwru_eda.ipynb
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\notebooks\05_cwru_fft_analysis.ipynb
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\notebooks\06_cwru_physics_features.ipynb
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\notebooks\07_cwru_feature_analysis_baseline.ipynb
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\notebooks\08_cwru_window_dataset.ipynb
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\notebooks\09_cwru_dataset_split_and_windows.ipynb
c:\U

In [66]:
notebook_files = sorted(
    (PROJECT_ROOT / "notebooks").glob("*.ipynb")
)

print("Notebook files:")
for path in notebook_files:
    print(path.name)

Notebook files:
01_cwru_exploration.ipynb
02_cwru_metadata.ipynb
03_cwru_signal_extraction.ipynb
04_cwru_eda.ipynb
05_cwru_fft_analysis.ipynb
06_cwru_physics_features.ipynb
07_cwru_feature_analysis_baseline.ipynb
08_cwru_window_dataset.ipynb
09_cwru_dataset_split_and_windows.ipynb
10_VAAC_Tiny_Algorithm_Development.ipynb
11_VAAC_Tiny_Fusion.ipynb
12_VAAC_Tiny_Training_Pipeline.ipynb
13_Baseline_Models.ipynb
14_Baseline_Models.ipynb
15_VAAC_Tiny_Training.ipynb
16_VAAC_Tiny_Temporal_Consensus.ipynb
17_VAAC_Tiny_Final_Analysis.ipynb
18_VAAC_Tiny_Optimization.ipynb
19_VAAC_Tiny_INT8_Quantization.ipynb
20_VAAC_Tiny_INT8_Accuracy_Verification.ipynb
21_VAAC_Tiny_Output_Mapping_Diagnosis.ipynb


In [67]:
import json

matches = []

for notebook_path in notebook_files:
    
    try:
        with open(
            notebook_path,
            "r",
            encoding="utf-8"
        ) as f:
            notebook = json.load(f)
        
        text = json.dumps(notebook)
        
        if "CLASS_NAMES" in text:
            matches.append(
                notebook_path.name
            )
            
    except Exception as e:
        print(
            "Could not read:",
            notebook_path.name,
            e
        )

print("Notebooks containing CLASS_NAMES:")
for name in matches:
    print("-", name)

Could not read: 20_VAAC_Tiny_INT8_Accuracy_Verification.ipynb Expecting value: line 1 column 1 (char 0)
Could not read: 21_VAAC_Tiny_Output_Mapping_Diagnosis.ipynb Expecting value: line 1 column 1 (char 0)
Notebooks containing CLASS_NAMES:
- 11_VAAC_Tiny_Fusion.ipynb
- 12_VAAC_Tiny_Training_Pipeline.ipynb
- 13_Baseline_Models.ipynb
- 14_Baseline_Models.ipynb
- 15_VAAC_Tiny_Training.ipynb
- 16_VAAC_Tiny_Temporal_Consensus.ipynb
- 17_VAAC_Tiny_Final_Analysis.ipynb


In [68]:
search_terms = [
    "Healthy",
    "Ball",
    "Inner Race",
    "Outer Race",
    "LabelEncoder",
    "class_names",
    "class_to",
    "label"
]

for notebook_path in notebook_files:
    
    try:
        with open(
            notebook_path,
            "r",
            encoding="utf-8"
        ) as f:
            notebook = json.load(f)
        
        text = json.dumps(
            notebook,
            ensure_ascii=False
        )
        
        found = [
            term
            for term in search_terms
            if term in text
        ]
        
        if found:
            print(
                notebook_path.name,
                "->",
                found
            )
            
    except:
        pass

01_cwru_exploration.ipynb -> ['Healthy', 'Ball', 'Inner Race', 'Outer Race']
02_cwru_metadata.ipynb -> ['Healthy', 'Ball', 'Inner Race', 'Outer Race']
04_cwru_eda.ipynb -> ['Healthy', 'Ball', 'Inner Race', 'Outer Race', 'label']
05_cwru_fft_analysis.ipynb -> ['Healthy', 'Ball', 'Inner Race', 'Outer Race', 'label']
06_cwru_physics_features.ipynb -> ['Healthy', 'Ball', 'Inner Race', 'Outer Race', 'label']
07_cwru_feature_analysis_baseline.ipynb -> ['Healthy', 'Ball', 'Inner Race', 'Outer Race', 'LabelEncoder', 'label']
08_cwru_window_dataset.ipynb -> ['Healthy', 'Ball', 'Inner Race', 'Outer Race']
09_cwru_dataset_split_and_windows.ipynb -> ['Healthy', 'Ball', 'Inner Race', 'Outer Race', 'label']
10_VAAC_Tiny_Algorithm_Development.ipynb -> ['Healthy', 'Ball', 'Inner Race', 'Outer Race', 'label']
11_VAAC_Tiny_Fusion.ipynb -> ['Healthy', 'Ball', 'Inner Race', 'Outer Race', 'label']
12_VAAC_Tiny_Training_Pipeline.ipynb -> ['Healthy', 'Ball', 'Inner Race', 'Outer Race', 'label']
13_Baseline_M

In [69]:
for notebook_path in notebook_files:
    
    try:
        with open(
            notebook_path,
            "r",
            encoding="utf-8"
        ) as f:
            notebook = json.load(f)
        
        text = json.dumps(notebook)
        
        if "LabelEncoder" in text:
            print(
                "LabelEncoder found in:",
                notebook_path.name
            )
            
    except:
        pass

LabelEncoder found in:

 07_cwru_feature_analysis_baseline.ipynb


In [71]:
# STEP 226.28-A
# Load train, validation and test metadata

TRAIN_METADATA = WINDOW_DIR / "train_metadata.csv"
VAL_METADATA = WINDOW_DIR / "validation_metadata.csv"
TEST_METADATA = WINDOW_DIR / "test_metadata.csv"

print("Train metadata exists:",
      TRAIN_METADATA.exists())

print("Validation metadata exists:",
      VAL_METADATA.exists())

print("Test metadata exists:",
      TEST_METADATA.exists())

train_df = pd.read_csv(TRAIN_METADATA)
val_df = pd.read_csv(VAL_METADATA)
test_df = pd.read_csv(TEST_METADATA)

print("\nTrain shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train metadata exists: True
Validation metadata exists: True
Test metadata exists: True

Train shape: (232, 8)
Validation shape: (215, 8)
Test shape: (136, 8)


In [72]:
print("Unique labels in training metadata:")

print(
    train_df["label"].value_counts().sort_index()
)

Unique labels in training metadata:
label
0    118
1     38
2     38
3     38
Name: count, dtype: int64


In [73]:
print("\nTRAIN CLASS → LABEL MAPPING")

print(
    train_df[
        ["class", "label"]
    ]
    .drop_duplicates()
    .sort_values("label")
    .to_string(index=False)
)


TRAIN CLASS → LABEL MAPPING
     class  label
   Healthy      0
      Ball      1
Inner Race      2
Outer Race      3


In [74]:
print("\nVALIDATION CLASS → LABEL MAPPING")

print(
    val_df[
        ["class", "label"]
    ]
    .drop_duplicates()
    .sort_values("label")
    .to_string(index=False)
)

print("\nTEST CLASS → LABEL MAPPING")

print(
    test_df[
        ["class", "label"]
    ]
    .drop_duplicates()
    .sort_values("label")
    .to_string(index=False)
)


VALIDATION CLASS → LABEL MAPPING
     class  label
   Healthy      0
      Ball      1
Inner Race      2
Outer Race      3

TEST CLASS → LABEL MAPPING
     class  label
   Healthy      0
      Ball      1
Inner Race      2
Outer Race      3


In [75]:
print(
    train_df[
        ["class", "label"]
    ].drop_duplicates().sort_values(
        "label"
    )
)

          class  label
38      Healthy      0
175        Ball      1
0    Inner Race      2
156  Outer Race      3


In [76]:
print("TRAIN:")
print(
    train_df[
        ["class", "label"]
    ].drop_duplicates().sort_values(
        "label"
    )
)

print("\nVALIDATION:")
print(
    val_df[
        ["class", "label"]
    ].drop_duplicates().sort_values(
        "label"
    )
)

print("\nTEST:")
print(
    test_df[
        ["class", "label"]
    ].drop_duplicates().sort_values(
        "label"
    )
)

TRAIN:
          class  label
38      Healthy      0
175        Ball      1
0    Inner Race      2
156  Outer Race      3

VALIDATION:
          class  label
0       Healthy      0
177        Ball      1
196  Inner Race      2
158  Outer Race      3

TEST:
         class  label
57     Healthy      0
0         Ball      1
19  Inner Race      2
38  Outer Race      3


In [77]:
train_mapping = (
    train_df[
        ["class", "label"]
    ]
    .drop_duplicates()
    .sort_values("class")
)

val_mapping = (
    val_df[
        ["class", "label"]
    ]
    .drop_duplicates()
    .sort_values("class")
)

test_mapping = (
    test_df[
        ["class", "label"]
    ]
    .drop_duplicates()
    .sort_values("class")
)

print("TRAIN == VALIDATION:",
      train_mapping.reset_index(drop=True).equals(
          val_mapping.reset_index(drop=True)
      ))

print("TRAIN == TEST:",
      train_mapping.reset_index(drop=True).equals(
          test_mapping.reset_index(drop=True)
      ))

TRAIN == VALIDATION: True
TRAIN == TEST: True


In [78]:
VERIFIED_CLASS_TO_LABEL = (
    train_df[
        ["class", "label"]
    ]
    .drop_duplicates()
    .set_index("class")["label"]
    .to_dict()
)

print(
    "VERIFIED CLASS → LABEL"
)

for class_name, label in sorted(
    VERIFIED_CLASS_TO_LABEL.items(),
    key=lambda x: x[1]
):
    print(
        f"{class_name:<15} = {label}"
    )

VERIFIED CLASS → LABEL
Healthy         = 0
Ball            = 1
Inner Race      = 2
Outer Race      = 3


In [79]:
y_true_verified = []

y_pred_model = []

for _, row in test_df.iterrows():
    
    path = get_window_path(row)
    
    signal = np.load(
        path
    ).astype(np.float32)
    
    signal = signal.reshape(
        1, 12000, 1
    )
    
    output = fp32_model.predict(
        signal,
        verbose=0
    )[0]
    
    predicted_label = int(
        np.argmax(output)
    )
    
    true_label = int(
        row["label"]
    )
    
    y_true_verified.append(
        true_label
    )
    
    y_pred_model.append(
        predicted_label
    )

In [80]:
verified_accuracy = accuracy_score(
    y_true_verified,
    y_pred_model
)

print(
    "VERIFIED FP32 ACCURACY:",
    f"{verified_accuracy:.4f}"
)

VERIFIED FP32 ACCURACY: 0.1397


In [81]:
verified_cm = confusion_matrix(
    y_true_verified,
    y_pred_model,
    labels=[0, 1, 2, 3]
)

print(
    "VERIFIED FP32 CONFUSION MATRIX"
)

print(verified_cm)

VERIFIED FP32 CONFUSION MATRIX
[[ 0  0  0 79]
 [ 0  0  0 19]
 [ 0 19  0  0]
 [ 0  0  0 19]]


In [82]:
print(
    classification_report(
        y_true_verified,
        y_pred_model,
        labels=[0, 1, 2, 3],
        target_names=[
            name
            for name, label
            in sorted(
                VERIFIED_CLASS_TO_LABEL.items(),
                key=lambda x: x[1]
            )
        ],
        zero_division=0
    )
)

              precision    recall  f1-score   support

     Healthy       0.00      0.00      0.00        79
        Ball       0.00      0.00      0.00        19
  Inner Race       0.00      0.00      0.00        19
  Outer Race       0.16      1.00      0.28        19

    accuracy                           0.14       136
   macro avg       0.04      0.25      0.07       136
weighted avg       0.02      0.14      0.04       136



In [83]:
print(
    train_df[
        ["class", "label"]
    ].drop_duplicates().sort_values("label")
)


          class  label
38      Healthy      0
175        Ball      1
0    Inner Race      2
156  Outer Race      3


In [84]:
print(
    "VERIFIED FP32 ACCURACY:",
    f"{verified_accuracy:.4f}"
)

VERIFIED FP32 ACCURACY: 0.1397
